# Chapter 2: Pre-Model Workflow and Data Preprocessing

**Reference:** John Sukup, *scikit-learn Cookbook*, Third Edition (2025).

This notebook follows the main topics of Chapter 2. The demonstrations are independently written and pair each code example with a short explanation of its output.

**Source:** John Sukup, *scikit-learn Cookbook*, Third Edition (Packt, 2025). The sections below follow the topics and selected examples presented in the book. Explanations and interpretations are summarized in original English.

## 1. The impact of raw data on model performance

Missing values, differing scales and categorical variables can cause otherwise suitable models to perform poorly. The first step is inspecting the dataset before estimating a model.

In [1]:
import numpy as np
import pandas as pd
raw = pd.DataFrame({'age':[21,34,np.nan,48,29],
                    'income':[2300,5100,3900,np.nan,3200],
                    'city':['A','B','A','C','B']})
print(raw)
print('\nMissing values:\n', raw.isna().sum())

    age  income city
0  21.0  2300.0    A
1  34.0  5100.0    B
2   NaN  3900.0    A
3  48.0     NaN    C
4  29.0  3200.0    B

Missing values:
 age       1
income    1
city      0
dtype: int64


**Interpretation**

The table contains missing entries in two numeric columns and text values in `city`. Those problems need different preprocessing techniques rather than a single blanket operation.

## 2. Handling missing data

`SimpleImputer` replaces absent observations with a statistic learned from available training data. Median imputation is often less affected by extreme values than mean imputation.

In [2]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
filled = imputer.fit_transform(raw[['age','income']])
print('Learned medians:', imputer.statistics_)
print('Imputed columns:\n', filled)

Learned medians: [  31.5 3550. ]
Imputed columns:
 [[  21.  2300. ]
 [  34.  5100. ]
 [  31.5 3900. ]
 [  48.  3550. ]
 [  29.  3200. ]]


**Interpretation**

Missing ages and incomes are replaced by their respective column medians. The other observations remain unchanged, preserving the originally recorded numeric values.

## 3. Scaling techniques

Standardization centers variables and rescales their spread; min–max scaling maps values into a specified interval. Their effects are different, especially when features contain extreme values.

In [3]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
values = np.array([[10,100],[20,250],[30,400],[40,1000]], dtype=float)
print('StandardScaler:\n', StandardScaler().fit_transform(values).round(2))
print('MinMaxScaler:\n', MinMaxScaler().fit_transform(values).round(2))

StandardScaler:
 [[-1.34 -0.99]
 [-0.45 -0.55]
 [ 0.45 -0.11]
 [ 1.34  1.65]]
MinMaxScaler:
 [[0.   0.  ]
 [0.33 0.17]
 [0.67 0.33]
 [1.   1.  ]]


**Interpretation**

StandardScaler produces values relative to the mean and standard deviation. MinMaxScaler maps each column to the interval from zero to one, leaving large gaps where the raw distribution is uneven.

## 4. Encoding categorical variables

One-hot encoding changes a nominal category into indicator columns without assigning an artificial ranking. Unknown values can be handled explicitly when the model is used later.

In [4]:
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoded = encoder.fit_transform(raw[['city']])
print('Categories:', encoder.categories_)
print('Encoded data:\n', encoded.astype(int))
print('Unseen city:\n', encoder.transform([['D']]).astype(int))

Categories: [array(['A', 'B', 'C'], dtype=object)]
Encoded data:
 [[1 0 0]
 [0 1 0]
 [1 0 0]
 [0 0 1]
 [0 1 0]]
Unseen city:
 [[0 0 0]]


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


**Interpretation**

Each known city activates its own column. The unseen city is represented by zeros under `handle_unknown="ignore"`, so inference does not fail when an unfamiliar category arrives.

## 5. Introduction to pipelines in scikit-learn

A `ColumnTransformer` processes numeric and categorical columns differently. Combining it with a pipeline helps ensure preprocessing is learned only from the training partition.

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
features = raw.copy()
features['city'] = features['city'].astype(str)
target = np.array([0.3,0.9,0.5,1.2,0.7])
pre = ColumnTransformer([
    ('numeric', Pipeline([('fill', SimpleImputer(strategy='median')),
                          ('scale', StandardScaler())]), ['age','income']),
    ('category', OneHotEncoder(handle_unknown='ignore'), ['city'])])
model = Pipeline([('preprocess', pre),('regression', Ridge())])
model.fit(features,target)
print('Training predictions:', model.predict(features).round(3))

Training predictions: [0.321 0.869 0.601 1.121 0.687]


**Interpretation**

Both imputation and scaling are executed inside the pipeline, while the categorical column is encoded separately. The example is deliberately small, so its predictions demonstrate the workflow rather than reliable accuracy.

## 6. Visualizing pipelines

Displaying a model structure can help verify that each transformer and the estimator are connected as intended. Text representation is practical when working in a notebook without interactive diagrams.

In [6]:
print(model)
print('Output feature count:', model.named_steps['preprocess'].transform(features).shape[1])

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('fill',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  ['age', 'income']),
                                                 ('category',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['city'])])),
                ('regression', Ridge())])
Output feature count: 5


**Interpretation**

The displayed pipeline shows the ordered preprocessing and prediction steps. The expanded feature count includes both scaled numeric variables and one-hot category columns.

## 7. Feature engineering

Feature engineering creates inputs that represent useful information not present directly in the raw observations. Polynomial terms and interactions can capture curved or combined effects.

In [7]:
from sklearn.preprocessing import PolynomialFeatures
X_base = np.array([[1,2],[2,3],[3,4]], dtype=float)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_base)
print('Features:', poly.get_feature_names_out(['a','b']))
print('New feature matrix:\n', X_poly)

Features: ['a' 'b' 'a^2' 'a b' 'b^2']
New feature matrix:
 [[ 1.  2.  1.  2.  4.]
 [ 2.  3.  4.  6.  9.]
 [ 3.  4.  9. 12. 16.]]


**Interpretation**

Two original variables expand into five terms: two linear terms, two squares, and one interaction. This allows a subsequent linear estimator to represent non-linear relationships, at the cost of more features.

## 8. Practical preprocessing checks

The same transformations should be reused for new records. Re-fitting preprocessing separately on a test record changes the reference statistics and invalidates the comparison.

In [8]:
new_record = pd.DataFrame({'age':[35.], 'income':[4200.], 'city':['B']})
print('Processed new record:', pre.transform(new_record).round(3))
print('Predicted target:', model.predict(new_record).round(3))

Processed new record: [[0.261 0.644 0.    1.    0.   ]]
Predicted target: [0.851]


**Interpretation**

The new observation is processed using the transformations already learned from training. This is the correct inference path for a pipeline that combines numeric and categorical preprocessing.

## Book recipe: Handling missing data — SimpleImputer, KNNImputer and IterativeImputer

The missing-value recipe in Chapter 2 compares three methods. `SimpleImputer` learns a column statistic, `KNNImputer` uses neighboring records, and `IterativeImputer` predicts missing features from the other columns. The following example reproduces the book’s comparison using a compact synthetic DataFrame.

In [9]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
np.random.seed(2024)
n_samples, n_features = 20, 10
data = {f'Feature{i+1}': np.random.uniform(0, 100, n_samples) for i in range(n_features)}
df_missing = pd.DataFrame(data)
for col in df_missing.columns:
    missing_rows = np.random.choice(df_missing.index, size=3, replace=False)
    df_missing.loc[missing_rows, col] = np.nan

simple_df = pd.DataFrame(SimpleImputer(strategy='mean').fit_transform(df_missing), columns=df_missing.columns)
knn_df = pd.DataFrame(KNNImputer(n_neighbors=2).fit_transform(df_missing), columns=df_missing.columns)
iter_df = pd.DataFrame(IterativeImputer(max_iter=10, random_state=2024).fit_transform(df_missing), columns=df_missing.columns)
print('Missing cells before:', int(df_missing.isna().sum().sum()))
print('Missing cells after each method:', *(int(d.isna().sum().sum()) for d in [simple_df, knn_df, iter_df]))
print(pd.DataFrame({'Original': df_missing.iloc[:6,0], 'Mean':simple_df.iloc[:6,0], 'KNN':knn_df.iloc[:6,0], 'Iterative':iter_df.iloc[:6,0]}))

Missing cells before: 30
Missing cells after each method: 0 0 0
    Original       Mean        KNN  Iterative
0  58.801452  58.801452  58.801452  58.801452
1  69.910875  69.910875  69.910875  69.910875
2        NaN  52.612877  71.599943  52.566772
3        NaN  52.612877  58.339826  52.879469
4  20.501895  20.501895  20.501895  20.501895
5  10.606287  10.606287  10.606287  10.606287


**Interpretation**

All three methods replace the missing entries, but they need not produce the same replacement value. Mean imputation ignores relationships between records; KNN and iterative imputation attempt to use additional information from the data.

## Book recipe: Scaling techniques — StandardScaler, MinMaxScaler and Normalizer

The book applies three scaling techniques to the imputed data. Standardization centers each feature, min–max scaling changes each feature’s range, and normalization rescales each individual sample.

In [10]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, Normalizer
standardized_df = pd.DataFrame(StandardScaler().fit_transform(iter_df), columns=iter_df.columns)
minmax_df = pd.DataFrame(MinMaxScaler().fit_transform(iter_df), columns=iter_df.columns)
normalized_df = pd.DataFrame(Normalizer().fit_transform(iter_df), columns=iter_df.columns)
print('Standardized means:', standardized_df.mean().round(5).tolist()[:4])
print('Min/Max of first feature:', round(float(minmax_df.iloc[:,0].min()),3), round(float(minmax_df.iloc[:,0].max()),3))
print('First three normalized row lengths:', np.linalg.norm(normalized_df.to_numpy(), axis=1)[:3].round(3))

Standardized means: [-0.0, 0.0, -0.0, -0.0]
Min/Max of first feature: 0.0 1.0
First three normalized row lengths: [1. 1. 1.]


**Interpretation**

The standardized feature means are approximately zero and the first min–max scaled feature runs from zero to one. Each normalized row has a length close to one. These methods therefore modify different statistical properties of the data.